# 10 - Prediction Only (Daily, Weekly, Monthly, Quarterly, Yearly)

This notebook is dedicated to **forward prediction only** with detailed outputs.

It provides no-leakage forecasts for these assets:
- S&P 500
- Nasdaq
- Gold
- Silver

For each asset, it predicts:
- Daily direction
- Weekly direction
- Monthly direction
- Quarterly direction
- Yearly direction

Label convention: `1 = UP (Close > Open)`, `0 = DOWN/FLAT`.

In [1]:
import os
import sys
from datetime import datetime, timedelta

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
import config

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

try:
    import yfinance as yf
except ModuleNotFoundError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'yfinance', '-q'])
    import yfinance as yf

np.random.seed(config.RANDOM_STATE)

In [10]:
def add_features(df, ma_short, ma_long, vol_window, rsi_window, bb_window):
    d = df.copy()
    d['Direction'] = (d['Close'] > d['Open']).astype(int)
    d['LogReturn'] = np.log(d['Close'] / d['Close'].shift(1))

    d['MA_short'] = d['Close'].rolling(ma_short).mean()
    d['MA_long'] = d['Close'].rolling(ma_long).mean()
    d['MA_cross'] = d['MA_short'] - d['MA_long']

    d['Volatility'] = d['LogReturn'].rolling(vol_window).std()

    delta = d['Close'].diff()
    gain = delta.clip(lower=0).rolling(rsi_window).mean()
    loss = (-delta.clip(upper=0)).rolling(rsi_window).mean()
    rs = gain / (loss + 1e-12)
    d['RSI'] = 100 - (100 / (1 + rs))

    bb_mid = d['Close'].rolling(bb_window).mean()
    bb_std = d['Close'].rolling(bb_window).std()
    d['BB_upper'] = bb_mid + config.BB_STD * bb_std
    d['BB_lower'] = bb_mid - config.BB_STD * bb_std
    d['BB_width'] = (d['BB_upper'] - d['BB_lower']) / bb_mid
    d['BB_pct'] = (d['Close'] - d['BB_lower']) / (d['BB_upper'] - d['BB_lower'])

    return d


def scaled_windows(divisor, min_w=2):
    ma_short = max(min_w, int(round(config.MA_SHORT / divisor)))
    ma_long = max(ma_short + 1, int(round(config.MA_LONG / divisor)))
    vol_w = max(min_w, int(round(config.VOL_WINDOW / divisor)))
    rsi_w = max(min_w, int(round(config.RSI_WINDOW / divisor)))
    bb_w = max(min_w, int(round(config.BB_WINDOW / divisor)))
    return ma_short, ma_long, vol_w, rsi_w, bb_w


FEATURES = ['MA_cross', 'Volatility', 'RSI', 'BB_width', 'BB_pct', 'LogReturn']

ASSETS = {
    'S&P 500': '^GSPC',
    'Nasdaq': '^IXIC',
    'Gold': 'GC=F',
    'Silver': 'SI=F',
}


def _fit_model(X_train, y_train):
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    model = RandomForestClassifier(
        n_estimators=config.N_ESTIMATORS,
        random_state=config.RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(X_train_sc, y_train)
    return model, scaler


def predict_daily(raw_daily, today):
    feat = add_features(
        raw_daily,
        config.MA_SHORT, config.MA_LONG, config.VOL_WINDOW, config.RSI_WINDOW, config.BB_WINDOW,
    )

    X = feat[FEATURES].shift(1)
    y = feat['Direction']

    train_mask = X.notna().all(axis=1) & y.notna() & (X.index.date < today.date())
    X_train = X.loc[train_mask]
    y_train = y.loc[train_mask].astype(int)
    if len(X_train) < 200:
        raise ValueError(f'Not enough daily training rows: {len(X_train)}')

    model, scaler = _fit_model(X_train, y_train)

    pred_date = today if today in X.index else X.index.max()
    x_now = X.loc[[pred_date]]
    if x_now.isna().any(axis=1).iloc[0]:
        raise ValueError(f'Daily feature row for {pred_date.date()} contains NaN values.')

    x_now_sc = scaler.transform(x_now)
    pred_label = int(model.predict(x_now_sc)[0])
    pred_proba_up = float(model.predict_proba(x_now_sc)[0, 1])

    return {
        'prediction_date': pred_date,
        'input_date': pred_date - pd.Timedelta(days=1),
        'pred_label': pred_label,
        'pred_proba_up': pred_proba_up,
        'open_price': float(raw_daily.loc[pred_date, 'Open']),
        'latest_price': float(raw_daily.loc[pred_date, 'Close']),
        'in_progress': pred_date.date() == today.date(),
    }


def predict_weekly(raw_daily, today):
    weekly = raw_daily.resample('W-FRI').agg({
        'Open': 'first',
        'High': 'max',
        'Low': 'min',
        'Close': 'last',
        'Volume': 'sum',
    }).dropna(subset=['Open', 'Close'])

    ma_s, ma_l, vol_w, rsi_w, bb_w = scaled_windows(divisor=5, min_w=2)
    feat = add_features(weekly, ma_s, ma_l, vol_w, rsi_w, bb_w)

    X = feat[FEATURES].shift(1)
    y = feat['Direction']

    days_to_friday = (4 - today.weekday()) % 7
    current_week_end = today + pd.Timedelta(days=days_to_friday)

    train_mask = X.notna().all(axis=1) & y.notna() & (X.index < current_week_end)
    X_train = X.loc[train_mask]
    y_train = y.loc[train_mask].astype(int)
    if len(X_train) < 60:
        raise ValueError(f'Not enough weekly training rows: {len(X_train)}')

    model, scaler = _fit_model(X_train, y_train)

    pred_date = current_week_end if current_week_end in X.index else X.index.max()
    x_now = X.loc[[pred_date]]
    if x_now.isna().any(axis=1).iloc[0]:
        raise ValueError(f'Weekly feature row for {pred_date.date()} contains NaN values.')

    x_now_sc = scaler.transform(x_now)
    pred_label = int(model.predict(x_now_sc)[0])
    pred_proba_up = float(model.predict_proba(x_now_sc)[0, 1])

    return {
        'prediction_date': pred_date,
        'input_date': pred_date - pd.Timedelta(days=7),
        'pred_label': pred_label,
        'pred_proba_up': pred_proba_up,
        'open_price': float(weekly.loc[pred_date, 'Open']),
        'latest_price': float(weekly.loc[pred_date, 'Close']),
        'in_progress': pred_date >= today,
    }


def predict_monthly(raw_daily, today):
    monthly = raw_daily.resample('ME').agg({
        'Open': 'first',
        'High': 'max',
        'Low': 'min',
        'Close': 'last',
        'Volume': 'sum',
    }).dropna(subset=['Open', 'Close'])

    ma_s, ma_l, vol_m, rsi_m, bb_m = scaled_windows(divisor=21, min_w=2)
    feat = add_features(monthly, ma_s, ma_l, vol_m, rsi_m, bb_m)

    X = feat[FEATURES].shift(1)
    y = feat['Direction']

    current_month_end = today.to_period('M').to_timestamp('M')

    train_mask = X.notna().all(axis=1) & y.notna() & (X.index < current_month_end)
    X_train = X.loc[train_mask]
    y_train = y.loc[train_mask].astype(int)
    if len(X_train) < 24:
        raise ValueError(f'Not enough monthly training rows: {len(X_train)}')

    model, scaler = _fit_model(X_train, y_train)

    pred_date = current_month_end if current_month_end in X.index else X.index.max()
    x_now = X.loc[[pred_date]]
    if x_now.isna().any(axis=1).iloc[0]:
        raise ValueError(f'Monthly feature row for {pred_date.date()} contains NaN values.')

    x_now_sc = scaler.transform(x_now)
    pred_label = int(model.predict(x_now_sc)[0])
    pred_proba_up = float(model.predict_proba(x_now_sc)[0, 1])

    prev_month_end = (pred_date.to_period('M') - 1).to_timestamp('M')

    return {
        'prediction_date': pred_date,
        'input_date': prev_month_end,
        'pred_label': pred_label,
        'pred_proba_up': pred_proba_up,
        'open_price': float(monthly.loc[pred_date, 'Open']),
        'latest_price': float(monthly.loc[pred_date, 'Close']),
        'in_progress': pred_date >= current_month_end,
    }


def predict_quarterly(raw_daily, today):
    quarterly = raw_daily.resample('QE').agg({
        'Open': 'first',
        'High': 'max',
        'Low': 'min',
        'Close': 'last',
        'Volume': 'sum',
    }).dropna(subset=['Open', 'Close'])

    ma_q, ma_l, vol_q, rsi_q, bb_q = scaled_windows(divisor=63, min_w=2)
    feat = add_features(quarterly, ma_q, ma_l, vol_q, rsi_q, bb_q)

    X = feat[FEATURES].shift(1)
    y = feat['Direction']

    current_quarter_end = today.to_period('Q').to_timestamp('Q')

    train_mask = X.notna().all(axis=1) & y.notna() & (X.index < current_quarter_end)
    X_train = X.loc[train_mask]
    y_train = y.loc[train_mask].astype(int)
    if len(X_train) < 40:
        raise ValueError(f'Not enough quarterly training rows: {len(X_train)}')

    model, scaler = _fit_model(X_train, y_train)

    pred_date = current_quarter_end if current_quarter_end in X.index else X.index.max()
    x_now = X.loc[[pred_date]]
    if x_now.isna().any(axis=1).iloc[0]:
        raise ValueError(f'Quarterly feature row for {pred_date.date()} contains NaN values.')

    x_now_sc = scaler.transform(x_now)
    pred_label = int(model.predict(x_now_sc)[0])
    pred_proba_up = float(model.predict_proba(x_now_sc)[0, 1])

    prev_quarter_end = (pred_date.to_period('Q') - 1).to_timestamp('Q')

    return {
        'prediction_date': pred_date,
        'input_date': prev_quarter_end,
        'pred_label': pred_label,
        'pred_proba_up': pred_proba_up,
        'open_price': float(quarterly.loc[pred_date, 'Open']),
        'latest_price': float(quarterly.loc[pred_date, 'Close']),
        'in_progress': pred_date >= current_quarter_end,
    }


def predict_yearly(raw_daily, today):
    yearly = raw_daily.resample('YE').agg({
        'Open': 'first',
        'High': 'max',
        'Low': 'min',
        'Close': 'last',
        'Volume': 'sum',
    }).dropna(subset=['Open', 'Close'])

    ma_y, ma_l, vol_y, rsi_y, bb_y = scaled_windows(divisor=252, min_w=2)
    feat = add_features(yearly, ma_y, ma_l, vol_y, rsi_y, bb_y)

    X = feat[FEATURES].shift(1)
    y = feat['Direction']

    current_year_end = today.to_period('Y').to_timestamp('Y')

    train_mask = X.notna().all(axis=1) & y.notna() & (X.index < current_year_end)
    X_train = X.loc[train_mask]
    y_train = y.loc[train_mask].astype(int)
    if len(X_train) < 10:
        raise ValueError(f'Not enough yearly training rows: {len(X_train)}')

    model, scaler = _fit_model(X_train, y_train)

    pred_date = current_year_end if current_year_end in X.index else X.index.max()
    x_now = X.loc[[pred_date]]
    if x_now.isna().any(axis=1).iloc[0]:
        raise ValueError(f'Yearly feature row for {pred_date.date()} contains NaN values.')

    x_now_sc = scaler.transform(x_now)
    pred_label = int(model.predict(x_now_sc)[0])
    pred_proba_up = float(model.predict_proba(x_now_sc)[0, 1])

    prev_year_end = (pred_date.to_period('Y') - 1).to_timestamp('Y')

    return {
        'prediction_date': pred_date,
        'input_date': prev_year_end,
        'pred_label': pred_label,
        'pred_proba_up': pred_proba_up,
        'open_price': float(yearly.loc[pred_date, 'Open']),
        'latest_price': float(yearly.loc[pred_date, 'Close']),
        'in_progress': pred_date >= current_year_end,
    }

In [3]:
today = pd.Timestamp(datetime.now().date())
fetch_end = (today + timedelta(days=1)).strftime('%Y-%m-%d')

market_data = {}
for asset_name, ticker in ASSETS.items():
    raw = yf.download(
        ticker,
        start=config.START_DATE,
        end=fetch_end,
        auto_adjust=True,
        progress=False,
    )

    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    if raw.empty:
        print(f'{asset_name:8s} ({ticker}) -> no data returned, skipping.')
        continue

    raw = raw.rename_axis('Date').sort_index()
    raw = raw[['Open', 'High', 'Low', 'Close', 'Volume']].dropna(subset=['Open', 'Close'])
    market_data[asset_name] = raw
    print(f'{asset_name:8s} ({ticker}) -> {len(raw):,} rows')

if not market_data:
    raise ValueError('No asset data was downloaded from yfinance.')

S&P 500  (^GSPC) -> 4,088 rows
Nasdaq   (^IXIC) -> 4,088 rows
Gold     (GC=F) -> 4,087 rows
Silver   (SI=F) -> 4,087 rows


## Daily Prediction

In [4]:
for asset_name, raw_daily in market_data.items():
    print('=' * 70)
    print(f'Asset                     : {asset_name}')

    try:
        out = predict_daily(raw_daily, today)
        print('Prediction horizon        : Daily')
        print(f"Prediction date           : {out['prediction_date'].date()}")
        print(f"Model input date          : {out['input_date'].date()} features (lag-1)")
        print(f"Predicted daily direction : {'UP' if out['pred_label'] == 1 else 'DOWN/FLAT'} ({out['pred_label']})")
        print(f"P(UP)                     : {out['pred_proba_up']:.3f}")
        print(f"Open price                : {out['open_price']:.2f}")
        print(f"Latest price (Close)      : {out['latest_price']:.2f}")
        if out['in_progress']:
            print('Note: For an open market session, "Close" may be intraday last price.')
    except Exception as exc:
        print(f'Daily prediction failed   : {exc}')

print('=' * 70)

Asset                     : S&P 500
Prediction horizon        : Daily
Prediction date           : 2026-04-06
Model input date          : 2026-04-05 features (lag-1)
Predicted daily direction : UP (1)
P(UP)                     : 0.850
Open price                : 6587.66
Latest price (Close)      : 6611.83
Asset                     : Nasdaq
Prediction horizon        : Daily
Prediction date           : 2026-04-06
Model input date          : 2026-04-05 features (lag-1)
Predicted daily direction : UP (1)
P(UP)                     : 0.850
Open price                : 21939.80
Latest price (Close)      : 21996.34
Asset                     : Gold
Prediction horizon        : Daily
Prediction date           : 2026-04-06
Model input date          : 2026-04-05 features (lag-1)
Predicted daily direction : DOWN/FLAT (0)
P(UP)                     : 0.330
Open price                : 4678.60
Latest price (Close)      : 4671.60
Asset                     : Silver
Prediction horizon        : Daily
Predicti

## Weekly Prediction

In [5]:
for asset_name, raw_daily in market_data.items():
    print('=' * 70)
    print(f'Asset                     : {asset_name}')

    try:
        out = predict_weekly(raw_daily, today)
        print('Prediction horizon        : Weekly')
        print(f"Predicted week ending     : {out['prediction_date'].date()}")
        print(f"Model input week ending   : {out['input_date'].date()} features (lag-1)")
        print(f"Predicted weekly direction: {'UP' if out['pred_label'] == 1 else 'DOWN/FLAT'} ({out['pred_label']})")
        print(f"P(UP)                     : {out['pred_proba_up']:.3f}")
        print(f"Week open price           : {out['open_price']:.2f}")
        print(f"Latest weekly close/last  : {out['latest_price']:.2f}")
        if out['in_progress']:
            print('Note: Current week is still in progress until Friday close.')
    except Exception as exc:
        print(f'Weekly prediction failed  : {exc}')

print('=' * 70)

Asset                     : S&P 500
Prediction horizon        : Weekly
Predicted week ending     : 2026-04-10
Model input week ending   : 2026-04-03 features (lag-1)
Predicted weekly direction: UP (1)
P(UP)                     : 0.790
Week open price           : 6587.66
Latest weekly close/last  : 6611.83
Note: Current week is still in progress until Friday close.
Asset                     : Nasdaq
Prediction horizon        : Weekly
Predicted week ending     : 2026-04-10
Model input week ending   : 2026-04-03 features (lag-1)
Predicted weekly direction: UP (1)
P(UP)                     : 0.590
Week open price           : 21939.80
Latest weekly close/last  : 21996.34
Note: Current week is still in progress until Friday close.
Asset                     : Gold
Prediction horizon        : Weekly
Predicted week ending     : 2026-04-10
Model input week ending   : 2026-04-03 features (lag-1)
Predicted weekly direction: UP (1)
P(UP)                     : 0.810
Week open price           : 4678.

## Monthly Prediction

In [ ]:
for asset_name, raw_daily in market_data.items():
    print('=' * 70)
    print(f'Asset                      : {asset_name}')

    try:
        out = predict_monthly(raw_daily, today)
        print('Prediction horizon         : Monthly')
        print(f"Predicted month ending     : {out['prediction_date'].date()}")
        print(f"Model input month ending   : {out['input_date'].date()} features (lag-1)")
        print(f"Predicted monthly direction: {'UP' if out['pred_label'] == 1 else 'DOWN/FLAT'} ({out['pred_label']})")
        print(f"P(UP)                      : {out['pred_proba_up']:.3f}")
        print(f"Month open price           : {out['open_price']:.2f}")
        print(f"Latest monthly close/last  : {out['latest_price']:.2f}")
        if out['in_progress']:
            print('Note: Current month is still in progress until month-end close.')
    except Exception as exc:
        print(f'Monthly prediction failed  : {exc}')

print('=' * 70)

Asset                      : S&P 500
Prediction horizon         : Monthly
Predicted month ending     : 2026-04-30
Model input month ending   : 2026-03-31 features (lag-1)
Predicted monthly direction: DOWN/FLAT (0)
P(UP)                      : 0.205
Month open price           : 6556.56
Latest monthly close/last  : 6611.83
Note: Current month is still in progress until month-end close.
Asset                      : Nasdaq
Prediction horizon         : Monthly
Predicted month ending     : 2026-04-30
Model input month ending   : 2026-03-31 features (lag-1)
Predicted monthly direction: UP (1)
P(UP)                      : 0.770
Month open price           : 21742.79
Latest monthly close/last  : 21996.34
Note: Current month is still in progress until month-end close.
Asset                      : Gold
Prediction horizon         : Monthly
Predicted month ending     : 2026-04-30
Model input month ending   : 2026-03-31 features (lag-1)
Predicted monthly direction: UP (1)
P(UP)                      :

## Quarterly Prediction

In [7]:
for asset_name, raw_daily in market_data.items():
    print('=' * 70)
    print(f'Asset                      : {asset_name}')

    try:
        out = predict_quarterly(raw_daily, today)
        print('Prediction horizon         : Quarterly')
        print(f"Predicted quarter ending   : {out['prediction_date'].date()}")
        print(f"Model input quarter ending : {out['input_date'].date()} features (lag-1)")
        print(f"Predicted quarterly direction: {'UP' if out['pred_label'] == 1 else 'DOWN/FLAT'} ({out['pred_label']})")
        print(f"P(UP)                      : {out['pred_proba_up']:.3f}")
        print(f"Quarter open price         : {out['open_price']:.2f}")
        print(f"Latest quarterly close/last: {out['latest_price']:.2f}")
        if out['in_progress']:
            print('Note: Current quarter is still in progress until quarter-end close.')
    except Exception as exc:
        print(f'Quarterly prediction failed: {exc}')

print('=' * 70)

Asset                      : S&P 500
Prediction horizon         : Quarterly
Predicted quarter ending   : 2026-06-30
Model input quarter ending : 2026-03-31 features (lag-1)
Predicted quarterly direction: UP (1)
P(UP)                      : 0.870
Quarter open price         : 6556.56
Latest quarterly close/last: 6611.83
Note: Current quarter is still in progress until quarter-end close.
Asset                      : Nasdaq
Prediction horizon         : Quarterly
Predicted quarter ending   : 2026-06-30
Model input quarter ending : 2026-03-31 features (lag-1)
Predicted quarterly direction: UP (1)
P(UP)                      : 0.880
Quarter open price         : 21742.79
Latest quarterly close/last: 21996.34
Note: Current quarter is still in progress until quarter-end close.
Asset                      : Gold
Prediction horizon         : Quarterly
Predicted quarter ending   : 2026-06-30
Model input quarter ending : 2026-03-31 features (lag-1)
Predicted quarterly direction: UP (1)
P(UP)          

## Yearly Prediction

In [11]:
for asset_name, raw_daily in market_data.items():
    print('=' * 70)
    print(f'Asset                      : {asset_name}')

    try:
        out = predict_yearly(raw_daily, today)
        print('Prediction horizon         : Yearly')
        print(f"Predicted year ending      : {out['prediction_date'].date()}")
        print(f"Model input year ending    : {out['input_date'].date()} features (lag-1)")
        print(f"Predicted yearly direction : {'UP' if out['pred_label'] == 1 else 'DOWN/FLAT'} ({out['pred_label']})")
        print(f"P(UP)                      : {out['pred_proba_up']:.3f}")
        print(f"Year open price            : {out['open_price']:.2f}")
        print(f"Latest yearly close/last   : {out['latest_price']:.2f}")
        if out['in_progress']:
            print('Note: Current year is still in progress until year-end close.')
    except Exception as exc:
        print(f'Yearly prediction failed   : {exc}')

print('=' * 70)

Asset                      : S&P 500
Prediction horizon         : Yearly
Predicted year ending      : 2026-12-31
Model input year ending    : 2025-12-31 features (lag-1)
Predicted yearly direction : UP (1)
P(UP)                      : 0.645
Year open price            : 6878.11
Latest yearly close/last   : 6611.83
Note: Current year is still in progress until year-end close.
Asset                      : Nasdaq
Prediction horizon         : Yearly
Predicted year ending      : 2026-12-31
Model input year ending    : 2025-12-31 features (lag-1)
Predicted yearly direction : UP (1)
P(UP)                      : 0.540
Year open price            : 23481.49
Latest yearly close/last   : 21996.34
Note: Current year is still in progress until year-end close.
Asset                      : Gold
Prediction horizon         : Yearly
Predicted year ending      : 2026-12-31
Model input year ending    : 2025-12-31 features (lag-1)
Predicted yearly direction : UP (1)
P(UP)                      : 0.565
Year op